# 02 — Gerar agents e aplicar questionário com LangChain

Este notebook lê a base `.csv` de agentes sintéticos, constrói um agente a partir de cada linha e aplica um questionário a uma notícia.

Esta versão usa **LangChain** com `ChatOpenAI`, mantendo o modelo `gpt-4o-mini`.

O desenho do agente segue a ordem:

1. `Persona`
2. variáveis socio-demográficas
3. variáveis de valores, atitudes e percepções
4. notícia/artefato
5. questionário aplicado como survey humano
6. resposta estruturada em JSON
7. agregação por constructo e por grupos de variáveis


## 1. Imports e configuração inicial

In [1]:
# Se necessário, rode uma vez:
# !pip install -U langchain langchain-openai langchain-core pandas numpy

from pathlib import Path
import os
import json
import time
import uuid
import pandas as pd
import numpy as np

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 240)


## 2. Caminhos do projeto

In [2]:
PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
INPUT_DIR = PROJECT_DIR / "data" / "input"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("INPUT_DIR:", INPUT_DIR)


PROJECT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm
OUTPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output
INPUT_DIR: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\input


## 3. LangChain + OpenAI

Configure antes a variável de ambiente `OPENAI_API_KEY`.

No PowerShell:

```powershell
setx OPENAI_API_KEY "sua-chave"
```

Depois feche e reabra o Jupyter/VS Code.


In [3]:
MODEL = "gpt-4o-mini"

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY não encontrada.")

llm = ChatOpenAI(
    model=MODEL,
    temperature=0.4,
)

parser = JsonOutputParser()

print("LangChain configurado com:", MODEL)


LangChain configurado com: gpt-4o-mini


## 4. Ler CSV da população de agentes

In [4]:
AGENTS_CSV = OUTPUT_DIR / "agents_population_n1000_seed42.csv"

if not AGENTS_CSV.exists():
    candidate_files = sorted(OUTPUT_DIR.glob("agents_population*.csv"))
    if candidate_files:
        AGENTS_CSV = candidate_files[0]

print("AGENTS_CSV:", AGENTS_CSV)
print("Existe?", AGENTS_CSV.exists())

df_agents = pd.read_csv(AGENTS_CSV)

print("Agentes carregados:", len(df_agents))
df_agents.head(3)


AGENTS_CSV: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\agents_population_n1000_seed42.csv
Existe? True
Agentes carregados: 1000


,id_persona,idade,gênero,escolaridade,situação ocupacional,estado civil,condição de moradia,tipo de moradia,localização,renda familiar mensal,acesso à internet,tamanho do domicílio,telefone,identificação política,ideologia,raça/cor,geração,segmento_e_bolhas,tipo_de_território,uf,tipo_de_fontes_de_informação,Religiosidade,Família,Honestidade,Meritocracia,Ideologia,Discriminação,Sexualidade,Comportamento_social,Identidade_nacional,Insegurança,Confiança,Persona,renda_pc
0,agent_2cc5d8ce623f,50-64 anos,Feminino,Ensino médio completo,Empregado com carteira assinada,Solteiro(a),Imóvel alugado,Apartamento,Capital,Mais de 20 salários mínimos,Tem internet fixa em casa,2-3 pessoas,Celular e telefone fixo,Centro-esquerda,Progressista,Parda,Geração Bossa Nova,Progressistas,Capital,RJ,Alta confiança histórica e institucional: jornais de referência; Digitais especializados ou econômicos,"Mede o quanto a fé organiza a vida moral e cotidiana. Considerar centralidade de Deus, prática religiosa e tolerância religiosa. Neste perfil, a religiosidade é interpretada pela combinação entre valoriza diversidade, minorias, clima, i...","Captura família como núcleo de pertencimento, proteção e motivação, incluindo tensão entre família tradicional e novos arranjos. A família tende a funcionar como rede de proteção e orientação moral, mas com abertura variável a novos arr...","Considera cumprimento de regras, informalidade, jeitinho e julgamento moral contextual. O julgamento moral tende a oscilar entre valorização de regras, sobrevivência cotidiana e tolerância à informalidade. O perfil de Progressistas e re...","Expressa crença em esforço individual, merecimento da ajuda, pobreza e apoio/rejeição a políticas redistributivas. A leitura sobre esforço pessoal e ajuda social é moldada por Progressistas, renda 'Mais de 1 SM' e geração Geração Bossa ...","Usar identidade política, conservadorismo/progressismo, estatismo econômico e punitivismo. A identidade política declarada é Centro-esquerda e a autoimagem ideológica é Progressista. Na prática, isso deve ser tratado como combinação ent...","Mede reconhecimento de racismo, machismo, preconceitos e ações afirmativas. A percepção sobre racismo, machismo e desigualdade deve considerar o lugar social de Feminino, Parda, renda 'Mais de 1 SM' e geração Geração Bossa Nova. O segme...","Captura aceitação de diversidade sexual, tolerância privada versus expressão pública e normas de gênero. A aceitação de diversidade sexual varia conforme geração e conservadorismo nos costumes. Como formada em um Brasil mais analógico, ...","Traduz sociabilidade cotidiana, hábitos de mídia, vida dentro/fora de casa e relação com segurança. A vida cotidiana combina circulação no território Capital, hábitos de mídia e sensação de segurança. O acesso à internet 'Tem internet f...","Mede orgulho/vergonha do Brasil, povo brasileiro, natureza, cultura regional e complexo de vira-lata. A identidade nacional tende a misturar orgulho do Brasil cultural/natural com crítica a corrupção, violência e desigualdade. O segment...","Captura medo da violência, vulnerabilidade percebida e tradução política do medo. A sensação de insegurança é calibrada por território, gênero, raça e renda. Neste perfil, Feminino, Parda, renda 'Mais de 1 SM' e moradia em Capital indic...","Mede confiança interpessoal, família, vizinhos, desconhecidos, instituições e pessoas diferentes. A confiança tende a ser maior em família e redes próximas, e menor em desconhecidos ou pessoas politicamente distantes. O segmento Progres...","Pessoa de 50-64 anos, Feminino, Parda, residente em RJ em Capital. Pertence à Geração Bossa Nova e ao segmento 'Progressistas'. Tem renda familiar mensal 'Mais de 20 salários mínimos', escolaridade 'Ensino médio completo' e situação ocu...",Mais de 1 SM
1,agent_29ef94c4ffbf,35-49 anos,Feminino,Ensino técnico / profissionalizante,Funcionário público,Casado(a),Imóvel cedido por familiar ou terceiros,Casa,Capital,Mais de 5 a 10 salários mínimos,Tem i

## 5. Definir blocos de variáveis do agente

A construção do agente começa pela `Persona`, porque ela funciona como a descrição narrativa principal.

Depois entram as variáveis socio-demográficas, que situam o agente socialmente.

Por fim entram as variáveis de pesquisa, que descrevem valores, atitudes, percepções e repertório informacional.


In [5]:
PERSONA_COLUMNS = ["Persona"]

SOCIO_DEMOGRAPHIC_COLUMNS = [
    "idade",
    "gênero",
    "escolaridade",
    "situação ocupacional",
    "estado civil",
    "condição de moradia",
    "tipo de moradia",
    "localização",
    "renda familiar mensal",
    "acesso à internet",
    "tamanho do domicílio",
    "telefone",
]

VALUES_ATTITUDES_PERCEPTIONS_COLUMNS = [
    "identificação política",
    "ideologia",
    "raça/cor",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
    "Religiosidade",
    "Família",
    "Honestidade",
    "Meritocracia",
    "Ideologia",
    "Discriminação",
    "Sexualidade",
    "Comportamento_social",
    "Identidade_nacional",
    "Insegurança",
    "Confiança",
]

ALL_AGENT_COLUMNS = (
    ["id_persona"]
    + PERSONA_COLUMNS
    + SOCIO_DEMOGRAPHIC_COLUMNS
    + VALUES_ATTITUDES_PERCEPTIONS_COLUMNS
)

missing = [c for c in ALL_AGENT_COLUMNS if c not in df_agents.columns]
print("Colunas ausentes:", missing)


Colunas ausentes: []


## 6. Construir contexto textual do agente

In [6]:
def safe_get(row: pd.Series, col: str, default: str = "") -> str:
    if col not in row.index:
        return default
    value = row[col]
    if pd.isna(value):
        return default
    return str(value)


def format_fields(row: pd.Series, columns: list[str]) -> str:
    return "\n".join(
        f"- {col}: {safe_get(row, col, 'Não informado')}"
        for col in columns
    )


def build_agent_context(row: pd.Series) -> str:
    id_persona = safe_get(row, "id_persona", "sem_id")
    persona = safe_get(row, "Persona", "Persona não informada.")
    socio = format_fields(row, SOCIO_DEMOGRAPHIC_COLUMNS)
    vap = format_fields(row, VALUES_ATTITUDES_PERCEPTIONS_COLUMNS)

    return f"""
ID DO AGENTE
- id_persona: {id_persona}

1. PERSONA

A variável Persona é a descrição narrativa principal do agente.
Ela sintetiza quem é essa pessoa, seu contexto social, seus hábitos, suas tensões e sua visão geral de mundo.
Use esta descrição como ponto de partida para interpretar todas as demais variáveis.

Persona:
{persona}

2. VARIÁVEIS SOCIO-DEMOGRÁFICAS

As variáveis socio-demográficas descrevem a posição social objetiva do agente.
Elas indicam idade, gênero, escolaridade, ocupação, estado civil, moradia, território, renda, acesso a meios de comunicação e composição domiciliar.
Essas variáveis devem orientar como o agente interpreta riscos, oportunidades, instituições, empresas, notícias e temas públicos.

{socio}

3. VALORES, ATITUDES E PERCEPÇÕES

As variáveis de pesquisa descrevem a visão de mundo do agente.
Elas incluem identificação política, ideologia, raça/cor, geração, segmento sociopolítico, território, UF, fontes de informação e dimensões psicográficas.
Essas variáveis devem orientar a resposta do agente em termos de crenças, confiança, insegurança, moralidade, família, religião, meritocracia, discriminação, sexualidade, identidade nacional e comportamento social.

{vap}
""".strip()


print(build_agent_context(df_agents.iloc[0])[:3000])


ID DO AGENTE
- id_persona: agent_2cc5d8ce623f

1. PERSONA

A variável Persona é a descrição narrativa principal do agente.
Ela sintetiza quem é essa pessoa, seu contexto social, seus hábitos, suas tensões e sua visão geral de mundo.
Use esta descrição como ponto de partida para interpretar todas as demais variáveis.

Persona:
Pessoa de 50-64 anos, Feminino, Parda, residente em RJ em Capital. Pertence à Geração Bossa Nova e ao segmento 'Progressistas'. Tem renda familiar mensal 'Mais de 20 salários mínimos', escolaridade 'Ensino médio completo' e situação ocupacional 'Empregado com carteira assinada'. Sua orientação política declarada é 'Centro-esquerda' e sua autoimagem ideológica é 'Progressista'.

Como agente sintético, deve responder a estímulos considerando uma combinação de dados demográficos, valores e atitudes. Sua visão de mundo combina valoriza diversidade, minorias, clima, igualdade e abertura de costumes, com maior proximidade de repertórios digitais e urbanos. e uma trajetó

## 7. Questionário

In [7]:
questionnaire = [
    {
        "question_id": "Q001",
        "construct": "confiança_institucional",
        "question": "Quanto você confia na informação apresentada nesta notícia?",
        "response_instruction": "Responda considerando sua confiança geral no conteúdo, na fonte e no modo como a notícia apresenta os fatos.",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Não confio nada",
            2: "Confio pouco",
            3: "Confio parcialmente",
            4: "Confio bastante",
            5: "Confio totalmente",
        },
    },
    {
        "question_id": "Q002",
        "construct": "percepção_de_risco",
        "question": "Quanto esta notícia aumenta sua sensação de risco, ameaça ou preocupação?",
        "response_instruction": "Responda considerando se a notícia parece indicar perigo, perda, injustiça, instabilidade, ameaça econômica, ameaça moral ou insegurança.",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Não aumenta nada",
            2: "Aumenta pouco",
            3: "Aumenta moderadamente",
            4: "Aumenta bastante",
            5: "Aumenta muito",
        },
    },
    {
        "question_id": "Q003",
        "construct": "probabilidade_de_compartilhamento",
        "question": "Qual é a chance de você compartilhar, comentar ou conversar sobre esta notícia?",
        "response_instruction": "Responda considerando seus hábitos de mídia, interesse no tema, identificação com a fonte e vontade de alertar, apoiar ou criticar o conteúdo.",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Muito improvável",
            2: "Pouco provável",
            3: "Talvez",
            4: "Provável",
            5: "Muito provável",
        },
    },
    {
        "question_id": "Q004",
        "construct": "valência_percebida",
        "question": "Qual é o tom geral que você percebe nesta notícia?",
        "response_instruction": "Responda considerando se a notícia parece negativa, crítica, neutra, favorável ou muito favorável ao tema apresentado.",
        "scale_min": 1,
        "scale_max": 5,
        "scale_labels": {
            1: "Muito negativo",
            2: "Negativo",
            3: "Neutro ou ambivalente",
            4: "Positivo",
            5: "Muito positivo",
        },
    },
]

df_questionnaire = pd.DataFrame(questionnaire)
QUESTIONNAIRE_CSV = INPUT_DIR / "questionnaire_langchain_example.csv"
df_questionnaire.to_csv(QUESTIONNAIRE_CSV, index=False, sep=";", encoding="utf-8-sig")

df_questionnaire


,question_id,construct,question,response_instruction,scale_min,scale_max,scale_labels
0,Q001,confiança_institucional,Quanto você confia na informação apresentada nesta notícia?,"Responda considerando sua confiança geral no conteúdo, na fonte e no modo como a notícia apresenta os fatos.",1,5,"{1: 'Não confio nada', 2: 'Confio pouco', 3: 'Confio parcialmente', 4: 'Confio bastante', 5: 'Confio totalmente'}"
1,Q002,percepção_de_risco,"Quanto esta notícia aumenta sua sensação de risco, ameaça ou preocupação?","Responda considerando se a notícia parece indicar perigo, perda, injustiça, instabilidade, ameaça econômica, ameaça moral ou insegurança.",1,5,"{1: 'Não aumenta nada', 2: 'Aumenta pouco', 3: 'Aumenta moderadamente', 4: 'Aumenta bastante', 5: 'Aumenta muito'}"
2,Q003,probabilidade_de_compartilhamento,"Qual é a chance de você compartilhar, comentar ou conversar sobre esta notícia?","Responda considerando seus hábitos de mídia, interesse no tema, identificação com a fonte e vontade de alertar, apoiar ou criticar o conteúdo.",1,5,"{1: 'Muito improvável', 2: 'Pouco provável', 3: 'Talvez', 4: 'Provável', 5: 'Muito provável'}"
3,Q004,valência_percebida,Qual é o tom geral que você percebe nesta notícia?,"Responda considerando se a notícia parece negativa, crítica, neutra, favorável ou muito favorável ao tema apresentado.",1,5,"{1: 'Muito negativo', 2: 'Negativo', 3: 'Neutro ou ambivalente', 4: 'Positivo', 5: 'Muito positivo'}"


## 8. Artefato de notícia com tipo de fonte

In [8]:
artifact = {
    "artifact_id": "NEWS_001",
    "artifact_type": "news_text",
    "title": "Empresa anuncia investimento em tecnologia e sustentabilidade",
    "tipo_de_fontes_de_informação": "Alta confiança histórica e institucional: jornais de referência",
    "source_name": "Jornal de referência",
    "content": """
Uma grande empresa brasileira anunciou um novo plano de investimento em tecnologia,
inteligência artificial e sustentabilidade. Segundo a companhia, a iniciativa deve aumentar
a eficiência operacional, reduzir emissões e abrir novas oportunidades de emprego qualificado.

Especialistas avaliam que o plano pode fortalecer a imagem institucional da empresa,
mas organizações da sociedade civil cobram mais transparência sobre metas ambientais,
governança dos dados e impactos sobre trabalhadores em funções operacionais.
""".strip(),
}

ARTIFACT_JSON = INPUT_DIR / "artifact_news_001_langchain.json"
ARTIFACT_JSON.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")

artifact


{'artifact_id': 'NEWS_001',
 'artifact_type': 'news_text',
 'title': 'Empresa anuncia investimento em tecnologia e sustentabilidade',
 'tipo_de_fontes_de_informação': 'Alta confiança histórica e institucional: jornais de referência',
 'source_name': 'Jornal de referência',
 'content': 'Uma grande empresa brasileira anunciou um novo plano de investimento em tecnologia,\ninteligência artificial e sustentabilidade. Segundo a companhia, a iniciativa deve aumentar\na eficiência operacional, reduzir emissões e abrir novas oportunidades de emprego qualificado.\n\nEspecialistas avaliam que o plano pode fortalecer a imagem institucional da empresa,\nmas organizações da sociedade civil cobram mais transparência sobre metas ambientais,\ngovernança dos dados e impactos sobre trabalhadores em funções operacionais.'}

## 9. Familiaridade com a fonte

A regra é:

- se a fonte da notícia está entre as fontes que o agente costuma consumir, a resposta tende a ser mais receptiva/otimista;
- se a fonte não é familiar, a resposta tende a ser mais cética/pessimista.

Isso não força a resposta. Apenas cria uma predisposição interpretativa.


In [9]:
def check_source_familiarity(agent_row: pd.Series, artifact: dict) -> dict:
    agent_sources = safe_get(agent_row, "tipo_de_fontes_de_informação", "")
    artifact_source = artifact.get("tipo_de_fontes_de_informação", "")

    is_familiar = artifact_source.lower() in agent_sources.lower()

    if is_familiar:
        instruction = """
O tipo de fonte desta notícia é familiar para o agente.
Como o agente costuma consumir esse tipo de fonte, ele tende a interpretar o conteúdo com maior familiaridade, maior abertura inicial e uma leitura relativamente mais otimista.
Essa orientação não deve forçar uma resposta positiva; ela apenas aumenta a predisposição de confiança e receptividade, desde que o conteúdo seja compatível com o perfil.
""".strip()
    else:
        instruction = """
O tipo de fonte desta notícia não é familiar para o agente.
Como o agente não costuma consumir esse tipo de fonte, ele tende a interpretar o conteúdo com maior distância, ceticismo e uma leitura relativamente mais pessimista.
Essa orientação não deve forçar uma resposta negativa; ela apenas aumenta a predisposição de cautela, dúvida ou menor confiança, de acordo com o perfil.
""".strip()

    return {
        "fonte_familiar": is_familiar,
        "agent_sources": agent_sources,
        "artifact_source": artifact_source,
        "source_alignment_instruction": instruction,
    }


check_source_familiarity(df_agents.iloc[0], artifact)


{'fonte_familiar': True,
 'agent_sources': 'Alta confiança histórica e institucional: jornais de referência; Digitais especializados ou econômicos',
 'artifact_source': 'Alta confiança histórica e institucional: jornais de referência',
 'source_alignment_instruction': 'O tipo de fonte desta notícia é familiar para o agente.\nComo o agente costuma consumir esse tipo de fonte, ele tende a interpretar o conteúdo com maior familiaridade, maior abertura inicial e uma leitura relativamente mais otimista.\nEssa orientação não deve forçar uma resposta positiva; ela apenas aumenta a predisposição de confiança e receptividade, desde que o conteúdo seja compatível com o perfil.'}

## 10. Prompt LangChain

In [10]:
agent_question_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Você é um simulador de respondentes sintéticos para um projeto de Synthetic Audiences baseado em Agent-Based Modeling e Cognitive Modeling.

Você deve responder como se fosse uma pessoa real participando de uma pesquisa.
Você não deve responder como especialista, consultor, pesquisador ou modelo de IA.

REGRAS GERAIS:
1. Leia primeiro a Persona.
2. Depois use as variáveis socio-demográficas.
3. Depois use as variáveis de valores, atitudes e percepções.
4. Responda como esse agente tenderia a responder.
5. Use a escala exatamente como apresentada.
6. Retorne apenas JSON válido.
7. Não exponha raciocínio passo a passo.
8. A justificativa deve ter no máximo duas frases.

FORMATO OBRIGATÓRIO DO JSON:
{{
  "answer_value": número inteiro dentro da escala,
  "answer_label": "rótulo exato da escala",
  "construct": "nome do constructo",
  "rationale": "justificativa curta em português",
  "confidence": número entre 0 e 1,
  "source_familiarity_effect": "como a familiaridade com a fonte influenciou a resposta",
  "agent_variable_reading": {{
    "persona_summary": "síntese curta de como a persona influenciou a resposta",
    "socio_demographic_summary": "síntese curta de como as variáveis socio-demográficas influenciaram a resposta",
    "values_attitudes_perceptions_summary": "síntese curta de como valores, atitudes e percepções influenciaram a resposta"
  }}
}}
""".strip(),
        ),
        (
            "human",
            """
CONTEXTO DO AGENTE

{agent_context}

CONTEXTO DA FONTE DA NOTÍCIA

Tipo de fonte da notícia:
{artifact_source}

Fontes que o agente costuma consumir:
{agent_sources}

A fonte da notícia é familiar para o agente?
{fonte_familiar}

Orientação sobre familiaridade com a fonte:
{source_alignment_instruction}

ARTEFATO ANALISADO

ID do artefato:
{artifact_id}

Tipo do artefato:
{artifact_type}

Título:
{artifact_title}

Conteúdo:
{artifact_content}

APLICAÇÃO DO QUESTIONÁRIO

Agora você será exposto a uma pergunta de questionário, como em uma pesquisa aplicada a seres humanos.

Instruções ao respondente:
- Leia a notícia.
- Responda à pergunta pensando na sua impressão pessoal.
- Não tente dar uma resposta tecnicamente correta.
- Não tente agradar o pesquisador.
- Responda de acordo com seus valores, experiências, hábitos de mídia e visão de mundo.
- Use apenas uma opção da escala.

Constructo medido:
{construct}

Pergunta:
{question}

Instrução específica de resposta:
{response_instruction}

Escala:
{scale_text}

Retorne apenas JSON válido.
""".strip(),
        ),
    ]
)

chain = agent_question_prompt | llm | parser


## 11. Função LangChain para aplicar uma pergunta a um agente

In [11]:
def scale_to_text(question: dict) -> str:
    return "\n".join(
        f"{k}: {v}" for k, v in question["scale_labels"].items()
    )


def run_agent_question(agent_row: pd.Series, question: dict, artifact: dict) -> dict:
    source_info = check_source_familiarity(agent_row, artifact)

    result = chain.invoke(
        {
            "agent_context": build_agent_context(agent_row),
            "artifact_source": artifact.get("tipo_de_fontes_de_informação", ""),
            "agent_sources": source_info["agent_sources"],
            "fonte_familiar": source_info["fonte_familiar"],
            "source_alignment_instruction": source_info["source_alignment_instruction"],
            "artifact_id": artifact.get("artifact_id", ""),
            "artifact_type": artifact.get("artifact_type", ""),
            "artifact_title": artifact.get("title", ""),
            "artifact_content": artifact.get("content", ""),
            "construct": question["construct"],
            "question": question["question"],
            "response_instruction": question["response_instruction"],
            "scale_text": scale_to_text(question),
        }
    )

    result["answer_value"] = int(result["answer_value"])
    result["confidence"] = float(result.get("confidence", 0.75))
    result["fonte_familiar"] = source_info["fonte_familiar"]
    result["artifact_source"] = source_info["artifact_source"]
    result["agent_sources"] = source_info["agent_sources"]

    return result


# Teste com 1 agente e 1 pergunta
test_result = run_agent_question(df_agents.iloc[0], questionnaire[0], artifact)
test_result


{'answer_value': 4,
 'answer_label': 'Confio bastante',
 'construct': 'confiança_institucional',
 'rationale': 'A fonte é familiar e de alta confiança, o que me faz acreditar na informação apresentada.',
 'confidence': 0.8,
 'source_familiarity_effect': 'A familiaridade com a fonte aumenta minha receptividade e confiança nas informações.',
 'agent_variable_reading': {'persona_summary': 'Como uma pessoa da Geração Bossa Nova, valorizo iniciativas que promovem tecnologia e sustentabilidade.',
  'socio_demographic_summary': 'Minha renda e educação me permitem ter uma visão crítica, mas confiante em informações de fontes respeitáveis.',
  'values_attitudes_perceptions_summary': 'Meu perfil progressista me faz apoiar investimentos em sustentabilidade, embora eu também exija transparência.'},
 'fonte_familiar': True,
 'artifact_source': 'Alta confiança histórica e institucional: jornais de referência',
 'agent_sources': 'Alta confiança histórica e institucional: jornais de referência; Digita

## 12. Rodar agentes

In [12]:
LIMIT_AGENTS = 10
REPEATS = 3
SLEEP_SECONDS = 0.2

RUN_ID = f"run_langchain_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"

RESPONSES_JSONL = OUTPUT_DIR / f"{RUN_ID}_responses.jsonl"
RESPONSES_CSV = OUTPUT_DIR / f"{RUN_ID}_responses.csv"
AGG_CONSTRUCT_CSV = OUTPUT_DIR / f"{RUN_ID}_aggregate_by_construct.csv"
AGG_VARIABLES_CSV = OUTPUT_DIR / f"{RUN_ID}_aggregate_by_variables_construct.csv"
AGG_AGENT_CONSTRUCT_CSV = OUTPUT_DIR / f"{RUN_ID}_aggregate_by_agent_construct.csv"

df_run_agents = df_agents.copy() if LIMIT_AGENTS is None else df_agents.head(LIMIT_AGENTS).copy()

total_calls = len(df_run_agents) * len(questionnaire) * REPEATS

print("RUN_ID:", RUN_ID)
print("Modelo:", MODEL)
print("Agentes:", len(df_run_agents))
print("Perguntas:", len(questionnaire))
print("Repeats:", REPEATS)
print("Total de chamadas:", total_calls)


RUN_ID: run_langchain_20260616_111806_2015b1
Modelo: gpt-4o-mini
Agentes: 10
Perguntas: 4
Repeats: 3
Total de chamadas: 120


In [13]:
records = []
errors = []
call_n = 0

with RESPONSES_JSONL.open("w", encoding="utf-8") as f:
    for _, agent_row in df_run_agents.iterrows():
        for question in questionnaire:
            for repeat_id in range(REPEATS):
                call_n += 1

                try:
                    answer = run_agent_question(agent_row, question, artifact)
                    variable_reading = answer.get("agent_variable_reading", {})

                    record = {
                        "run_id": RUN_ID,
                        "model": MODEL,
                        "artifact_id": artifact["artifact_id"],
                        "artifact_type": artifact["artifact_type"],
                        "artifact_title": artifact["title"],
                        "artifact_tipo_de_fontes_de_informação": artifact["tipo_de_fontes_de_informação"],

                        "id_persona": safe_get(agent_row, "id_persona"),
                        "Persona": safe_get(agent_row, "Persona"),

                        **{col: safe_get(agent_row, col) for col in SOCIO_DEMOGRAPHIC_COLUMNS},
                        **{col: safe_get(agent_row, col) for col in VALUES_ATTITUDES_PERCEPTIONS_COLUMNS},

                        "question_id": question["question_id"],
                        "construct": question["construct"],
                        "question": question["question"],
                        "repeat_id": repeat_id,

                        "answer_value": answer.get("answer_value"),
                        "answer_label": answer.get("answer_label"),
                        "rationale": answer.get("rationale"),
                        "confidence": answer.get("confidence"),
                        "fonte_familiar": answer.get("fonte_familiar"),
                        "source_familiarity_effect": answer.get("source_familiarity_effect"),

                        "persona_summary": variable_reading.get("persona_summary"),
                        "socio_demographic_summary": variable_reading.get("socio_demographic_summary"),
                        "values_attitudes_perceptions_summary": variable_reading.get("values_attitudes_perceptions_summary"),

                        "raw_answer": json.dumps(answer, ensure_ascii=False),
                    }

                    records.append(record)
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")

                except Exception as e:
                    error_record = {
                        "call_n": call_n,
                        "id_persona": safe_get(agent_row, "id_persona"),
                        "question_id": question["question_id"],
                        "repeat_id": repeat_id,
                        "error": repr(e),
                    }
                    errors.append(error_record)
                    print("ERRO:", error_record)

                if call_n % 10 == 0:
                    print(f"Progresso: {call_n}/{total_calls}")

                time.sleep(SLEEP_SECONDS)

df_responses = pd.DataFrame(records)
df_responses.to_csv(RESPONSES_CSV, index=False, sep=";", encoding="utf-8-sig")

print("Finalizado.")
print("Respostas válidas:", len(df_responses))
print("Erros:", len(errors))
print("CSV salvo em:", RESPONSES_CSV)

df_responses.head()


Progresso: 10/120
Progresso: 20/120
Progresso: 30/120
Progresso: 40/120
Progresso: 50/120
Progresso: 60/120
Progresso: 70/120
Progresso: 80/120
Progresso: 90/120
Progresso: 100/120
Progresso: 110/120
Progresso: 120/120
Finalizado.
Respostas válidas: 120
Erros: 0
CSV salvo em: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_responses.csv


,run_id,model,artifact_id,artifact_type,artifact_title,artifact_tipo_de_fontes_de_informação,id_persona,Persona,idade,gênero,escolaridade,situação ocupacional,estado civil,condição de moradia,tipo de moradia,localização,renda familiar mensal,acesso à internet,tamanho do domicílio,telefone,identificação política,ideologia,raça/cor,geração,segmento_e_bolhas,tipo_de_território,uf,tipo_de_fontes_de_informação,Religiosidade,Família,Honestidade,Meritocracia,Ideologia,Discriminação,Sexualidade,Comportamento_social,Identidade_nacional,Insegurança,Confiança,question_id,construct,question,repeat_id,answer_value,answer_label,rationale,confidence,fonte_familiar,source_familiarity_effect,persona_summary,socio_demographic_summary,values_attitudes_perceptions_summary,raw_answer
0,run_langchain_20260616_111806_2015b1,gpt-4o-mini,NEWS_001,news_text,Empresa anuncia investimento em tecnologia e sustentabilidade,Alta confiança histórica e institucional: jornais de referência,agent_2cc5d8ce623f,"Pessoa de 50-64 anos, Feminino, Parda, residente em RJ em Capital. Pertence à Geração Bossa Nova e ao segmento 'Progressistas'. Tem renda familiar mensal 'Mais de 20 salários mínimos', escolaridade 'Ensino médio completo' e situação ocu...",50-64 anos,Feminino,Ensino médio completo,Empregado com carteira assinada,Solteiro(a),Imóvel alugado,Apartamento,Capital,Mais de 20 salários mínimos,Tem internet fixa em casa,2-3 pessoas,Celular e telefone fixo,Centro-esquerda,Progressista,Parda,Geração Bossa Nova,Progressistas,Capital,RJ,Alta confiança histórica e institucional: jornais de referência; Digitais especializados ou econômicos,"Mede o quanto a fé organiza a vida moral e cotidiana. Considerar centralidade de Deus, prática religiosa e tolerância religiosa. Neste perfil, a religiosidade é interpretada pela combinação entre valoriza diversidade, minorias, clima, i...","Captura família como núcleo de pertencimento, proteção e motivação, incluindo tensão entre família tradicional e novos arranjos. A família tende a funcionar como rede de proteção e orientação moral, mas com abertura variável a novos arr...","Considera cumprimento de regras, informalidade, jeitinho e julgamento moral contextual. O julgamento moral tende a oscilar entre valorização de regras, sobrevivência cotidiana e tolerância à informalidade. O perfil de Progressistas e re...","Expressa crença em esforço individual, merecimento da ajuda, pobreza e apoio/rejeição a políticas redistributivas. A leitura sobre esforço pessoal e ajuda social é moldada por Progressistas, renda 'Mais de 1 SM' e geração Geração Bossa ...","Usar identidade política, conservadorismo/progressismo, estatismo econômico e punitivismo. A identidade política declarada é Centro-esquerda e a autoimagem ideológica é Progressista. Na prática, isso deve ser tratado como combinação ent...","Mede reconhecimento de racismo, machismo, preconceitos e ações afirmativas. A percepção sobre racismo, machismo e desigualdade deve considerar o lugar social de Feminino, Parda, renda 'Mais de 1 SM' e geração Geração Bossa Nova. O segme...","Captura aceitação de diversidade sexual, tolerância privada versus expressão pública e normas de gênero. A aceitação de diversidade sexual varia conforme geração e conservadorismo nos costumes. Como formada em um Brasil mais analógico, ...","Traduz sociabilidade cotidiana, hábitos de mídia, vida dentro/fora de casa e relação com segurança. A vida cotidiana combina circulação no território Capital, hábitos de mídia e sensação de segurança. O acesso à internet 'Tem internet f...","Mede orgulho/vergonha do Brasil, povo brasileiro, natureza, cultura regional e complexo de vira-lata. A identidade nacional tende a misturar orgulho do Brasil cultural/natural com crítica a corrupção, violência e desigualdade. O segment...","Captura medo da violência, vulnerabilidade percebida e tradução política do medo. A sensação de insegurança é calibrada por território, gênero, raça e renda. Neste perfil, Feminino, Parda, renda 'M

## 13. Média por constructo

In [14]:
df_responses["answer_value"] = pd.to_numeric(df_responses["answer_value"], errors="coerce")
df_responses["confidence"] = pd.to_numeric(df_responses["confidence"], errors="coerce")

df_construct = (
    df_responses
    .groupby(["artifact_id", "construct"], as_index=False)
    .agg(
        mean_answer=("answer_value", "mean"),
        std_answer=("answer_value", "std"),
        median_answer=("answer_value", "median"),
        n_responses=("answer_value", "count"),
        n_agents=("id_persona", "nunique"),
        mean_confidence=("confidence", "mean"),
    )
)

for col in ["mean_answer", "std_answer", "median_answer", "mean_confidence"]:
    df_construct[col] = df_construct[col].round(3)

df_construct.to_csv(AGG_CONSTRUCT_CSV, index=False, sep=";", encoding="utf-8-sig")

df_construct


,artifact_id,construct,mean_answer,std_answer,median_answer,n_responses,n_agents,mean_confidence
0,NEWS_001,confiança_institucional,3.000,1.017,3.0,30,10,0.750
1,NEWS_001,percepção_de_risco,2.733,0.691,3.0,30,10,0.713
2,NEWS_001,probabilidade_de_compartilhamento,3.000,1.017,3.0,30,10,0.750
3,NEWS_001,valência_percebida,3.533,0.507,4.0,30,10,0.750


## 14. Média por constructo e por conjunto de variáveis

In [15]:
GROUP_VARIABLES = [
    "segmento_e_bolhas",
    "geração",
    "ideologia",
    "identificação política",
    "raça/cor",
    "gênero",
    "renda familiar mensal",
    "uf",
    "tipo_de_território",
    "tipo_de_fontes_de_informação",
    "fonte_familiar",
]

agg_tables = []

for group_var in GROUP_VARIABLES:
    if group_var not in df_responses.columns:
        continue

    temp = (
        df_responses
        .groupby(["artifact_id", "construct", group_var], as_index=False)
        .agg(
            mean_answer=("answer_value", "mean"),
            std_answer=("answer_value", "std"),
            median_answer=("answer_value", "median"),
            n_responses=("answer_value", "count"),
            n_agents=("id_persona", "nunique"),
            mean_confidence=("confidence", "mean"),
        )
        .rename(columns={group_var: "group_value"})
    )

    temp["group_variable"] = group_var
    agg_tables.append(temp)

df_variables_construct = pd.concat(agg_tables, ignore_index=True)

for col in ["mean_answer", "std_answer", "median_answer", "mean_confidence"]:
    df_variables_construct[col] = df_variables_construct[col].round(3)

df_variables_construct = df_variables_construct[
    [
        "artifact_id",
        "construct",
        "group_variable",
        "group_value",
        "mean_answer",
        "std_answer",
        "median_answer",
        "n_responses",
        "n_agents",
        "mean_confidence",
    ]
]

df_variables_construct.to_csv(AGG_VARIABLES_CSV, index=False, sep=";", encoding="utf-8-sig")

df_variables_construct.head(30)


,artifact_id,construct,group_variable,group_value,mean_answer,std_answer,median_answer,n_responses,n_agents,mean_confidence
0,NEWS_001,confiança_institucional,segmento_e_bolhas,Agro,2.000,0.000,2.0,3,1,0.700
1,NEWS_001,confiança_institucional,segmento_e_bolhas,Conservadores cristãos,2.000,0.000,2.0,6,2,0.700
2,NEWS_001,confiança_institucional,segmento_e_bolhas,Dependentes do Estado / Classe D e E,2.000,0.000,2.0,6,2,0.700
3,NEWS_001,confiança_institucional,segmento_e_bolhas,Empresários,4.000,0.000,4.0,3,1,0.800
4,NEWS_001,confiança_institucional,segmento_e_bolhas,Liberais sociais,4.000,0.000,4.0,6,2,0.800
5,NEWS_001,confiança_institucional,segmento_e_bolhas,Progressistas,4.000,0.000,4.0,6,2,0.800
6,NEWS_001,percepção_de_risco,segmento_e_bolhas,Agro,3.000,0.000,3.0,3,1,0.700
7,NEWS_001,percepção_de_risco,segmento_e_bolhas,Conservadores cristãos,3.000,0.000,3.0,6,2,0.700
8,NEWS_001,percepção_de_risco,segmento_e_bolhas,Dependentes do Estado / Classe D e E,3.667,0.516,4.0,6,2,0.700
9,NEWS_001,percepção_de_risco,segmento_e_bolhas,Empresários,2.000,0.000,2.0,3,1,0.700


## 15. Média por agente e constructo

In [16]:
agent_group_cols = [
    "id_persona",
    "artifact_id",
    "construct",
    "segmento_e_bolhas",
    "geração",
    "ideologia",
    "identificação política",
    "raça/cor",
    "gênero",
    "renda familiar mensal",
    "uf",
    "tipo_de_território",
    "fonte_familiar",
]

df_agent_construct = (
    df_responses
    .groupby(agent_group_cols, as_index=False)
    .agg(
        mean_answer=("answer_value", "mean"),
        std_answer=("answer_value", "std"),
        n_repeats=("answer_value", "count"),
        mean_confidence=("confidence", "mean"),
    )
)

for col in ["mean_answer", "std_answer", "mean_confidence"]:
    df_agent_construct[col] = df_agent_construct[col].round(3)

df_agent_construct.to_csv(AGG_AGENT_CONSTRUCT_CSV, index=False, sep=";", encoding="utf-8-sig")

df_agent_construct.head(30)


,id_persona,artifact_id,construct,segmento_e_bolhas,geração,ideologia,identificação política,raça/cor,gênero,renda familiar mensal,uf,tipo_de_território,fonte_familiar,mean_answer,std_answer,n_repeats,mean_confidence
0,agent_29ef94c4ffbf,NEWS_001,confiança_institucional,Conservadores cristãos,Geração Redemocratização,Muito conservador(a),Centro,Branca,Feminino,Mais de 5 a 10 salários mínimos,SP,Capital,False,2.000,0.000,3,0.700
1,agent_29ef94c4ffbf,NEWS_001,percepção_de_risco,Conservadores cristãos,Geração Redemocratização,Muito conservador(a),Centro,Branca,Feminino,Mais de 5 a 10 salários mínimos,SP,Capital,False,3.000,0.000,3,0.700
2,agent_29ef94c4ffbf,NEWS_001,probabilidade_de_compartilhamento,Conservadores cristãos,Geração Redemocratização,Muito conservador(a),Centro,Branca,Feminino,Mais de 5 a 10 salários mínimos,SP,Capital,False,2.000,0.000,3,0.700
3,agent_29ef94c4ffbf,NEWS_001,valência_percebida,Conservadores cristãos,Geração Redemocratização,Muito conservador(a),Centro,Branca,Feminino,Mais de 5 a 10 salários mínimos,SP,Capital,False,3.333,0.577,3,0.700
4,agent_2cc5d8ce623f,NEWS_001,confiança_institucional,Progressistas,Geração Bossa Nova,Progressista,Centro-esquerda,Parda,Feminino,Mais de 20 salários mínimos,RJ,Capital,True,4.000,0.000,3,0.800
5,agent_2cc5d8ce623f,NEWS_001,percepção_de_risco,Progressistas,Geração Bossa Nova,Progressista,Centro-esquerda,Parda,Feminino,Mais de 20 salários mínimos,RJ,Capital,True,2.333,0.577,3,0.767
6,agent_2cc5d8ce623f,NEWS_001,probabilidade_de_compartilhamento,Progressistas,Geração Bossa Nova,Progressista,Centro-esquerda,Parda,Feminino,Mais de 20 salários mínimos,RJ,Capital,True,4.000,0.000,3,0.800
7,agent_2cc5d8ce623f,NEWS_001,valência_percebida,Progressistas,Geração Bossa Nova,Progressista,Centro-esquerda,Parda,Feminino,Mais de 20 salários mínimos,RJ,Capital,True,4.000,0.000,3,0.800
8,agent_5796645ef0d7,NEWS_001,confiança_institucional,Conservadores cristãos,Geração Ordem e Progresso,Conservador(a),Direita,Parda,Masculino,Até 1 salário mínimo,MG,Cidade média,False,2.000,0.000,3,0.700
9,agent_5796645ef0d7,NEWS_001,percepção_de_risco,Conservadores cristãos,Geração Ordem e Progresso,Conservador(a),Direita,Parda,Masculino,Até 1 salário mínimo,MG,Cidade média,False,3.000,0.000,3,0.700


## 16. Pivot executivo: segmento × constructo

In [17]:
pivot_segment_construct = (
    df_variables_construct[df_variables_construct["group_variable"] == "segmento_e_bolhas"]
    .pivot_table(
        index="group_value",
        columns="construct",
        values="mean_answer",
        aggfunc="mean",
    )
    .round(2)
)

PIVOT_SEGMENT_CSV = OUTPUT_DIR / f"{RUN_ID}_pivot_segment_construct.csv"
pivot_segment_construct.to_csv(PIVOT_SEGMENT_CSV, sep=";", encoding="utf-8-sig")

pivot_segment_construct


construct,confiança_institucional,percepção_de_risco,probabilidade_de_compartilhamento,valência_percebida
group_value,,,,
Agro,2.0,3.00,2.0,3.00
Conservadores cristãos,2.0,3.00,2.0,3.17
Dependentes do Estado / Classe D e E,2.0,3.67,2.0,3.00
Empresários,4.0,2.00,4.0,4.00
Liberais sociais,4.0,2.00,4.0,4.00
Progressistas,4.0,2.50,4.0,4.00


## 17. Auditoria qualitativa

In [18]:
audit_cols = [
    "id_persona",
    "segmento_e_bolhas",
    "geração",
    "ideologia",
    "construct",
    "answer_value",
    "answer_label",
    "fonte_familiar",
    "rationale",
    "source_familiarity_effect",
    "persona_summary",
    "socio_demographic_summary",
    "values_attitudes_perceptions_summary",
]

df_responses[audit_cols].sample(min(10, len(df_responses)), random_state=42)


,id_persona,segmento_e_bolhas,geração,ideologia,construct,answer_value,answer_label,fonte_familiar,rationale,source_familiarity_effect,persona_summary,socio_demographic_summary,values_attitudes_perceptions_summary
44,agent_920da99f9f9b,Progressistas,Geração Ordem e Progresso,Progressista,probabilidade_de_compartilhamento,4,Provável,True,"A notícia aborda temas que valorizo, como sustentabilidade e tecnologia, e a fonte é confiável.",A familiaridade com a fonte me faz interpretar a notícia de forma mais positiva e aberta.,"Como um progressista de 50-64 anos, valorizo iniciativas que promovem sustentabilidade e inovação.",Minha renda e educação me permitem ter interesse em notícias sobre tecnologia e sustentabilidade.,"Acredito na importância da transparência e na responsabilidade das empresas, o que me motiva a discutir essa notícia."
47,agent_920da99f9f9b,Progressistas,Geração Ordem e Progresso,Progressista,valência_percebida,4,Positivo,True,"A notícia apresenta um investimento em tecnologia e sustentabilidade, o que é favorável e alinhado aos meus valores progressistas.",A familiaridade com a fonte me faz confiar mais na informação apresentada.,"Como um progressista, valorizo iniciativas que promovem sustentabilidade e inovação.",Minha situação financeira e ocupacional me permite ver oportunidades em investimentos que geram empregos qualificados.,"Acredito que a tecnologia pode ser uma força positiva, especialmente quando associada à sustentabilidade e à responsabilidade social."
4,agent_2cc5d8ce623f,Progressistas,Geração Bossa Nova,Progressista,percepção_de_risco,3,Aumenta moderadamente,True,"Embora a iniciativa em tecnologia e sustentabilidade seja positiva, a falta de transparência sobre as metas e impactos gera preocupações.","A familiaridade com a fonte aumenta a confiança na informação, mas a crítica da sociedade civil provoca insegurança.","A persona valoriza iniciativas sustentáveis, mas é crítica em relação à transparência e impactos sociais.","A situação de renda alta e a ocupação estável influenciam uma visão crítica, mas esperançosa sobre investimentos.",Os valores progressistas e a preocupação com a justiça social elevam a percepção de risco diante da falta de clareza sobre as consequências do investimento.
55,agent_7cf1aaca19dd,Liberais sociais,Geração Redemocratização,Moderado(a),probabilidade_de_compartilhamento,4,Provável,True,"A notícia aborda um tema relevante sobre tecnologia e sustentabilidade, que me interessa e se alinha aos meus valores. Além disso, a fonte é confiável, o que me motiva a compartilhar.","A familiaridade com a fonte me deixou mais receptiva à mensagem, aumentando a probabilidade de compartilhar.","Como uma pessoa da Geração Redemocratização, valorizo inovações que promovem sustentabilidade e eficiência, o que me faz ver valor na notícia.","Minha situação ocupacional como aposentada me permite ter tempo para discutir temas relevantes, especialmente sobre investimentos que impactam o futuro.",Meus valores liberais sociais e a confiança em fontes institucionais me levam a apoiar iniciativas que buscam melhorias sociais e ambientais.
26,agent_ac2595547fe5,Empresários,Geração Ordem e Progresso,Conservador(a),confiança_institucional,4,Confio bastante,True,A notícia vem de uma fonte confiável e aborda um tema relevante para o meu contexto empresarial.,A familiaridade com a fonte aumenta minha confiança na informação apresentada.,"Como empresária, valorizo investimentos em tecnologia e sustentabilidade, o que me faz confiar na notícia.",Minha posição como empresária e residente em uma capital me leva a valorizar inovações que podem impactar positivamente o mercado.,"Meu conservadorismo e foco em eficiência me fazem ver a iniciativa com otimismo, apesar das críticas sobre transparência."
64,agent_7e4fb86f81fc,Dependentes do Estado / Classe D e E,Geração.Com,Conservador(a),percepção_de_risco,3,Aumenta moderadamente,False,"A notícia traz um investimento que pode ser positivo, mas a falta de transpa

## 18. Arquivos gerados

In [19]:
created_files = [
    RESPONSES_JSONL,
    RESPONSES_CSV,
    AGG_CONSTRUCT_CSV,
    AGG_VARIABLES_CSV,
    AGG_AGENT_CONSTRUCT_CSV,
    PIVOT_SEGMENT_CSV,
]

for file in created_files:
    print(file)


C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_responses.jsonl
C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_responses.csv
C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_aggregate_by_construct.csv
C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_aggregate_by_variables_construct.csv
C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_aggregate_by_agent_construct.csv
C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\output\run_langchain_20260616_111806_2015b1_pivot_segment_construct.csv
